In [ ]:
import os, glob, shutil, subprocess

WORK_DIR    = "/kaggle/working"
DATA8K_DIR  = f"{WORK_DIR}/data8k"
SCP_DIR     = f"{WORK_DIR}/scp"
RESUME_ROOT = f"{WORK_DIR}/resume_dir"
CKPT_NAME   = "Conv-TasNet-finetune"   

for d in [DATA8K_DIR, SCP_DIR, RESUME_ROOT]:
    os.makedirs(d, exist_ok=True)

GITHUB_REPO_URL = "https://github.com/JusperLee/Conv-TasNet.git"
CKPT_URL = "https://huggingface.co/KhaBui/PESEM-VS/resolve/main/Conv-TasNet_EN.pt"
REPO_ROOT_WRITABLE = f"{WORK_DIR}/Conv-TasNet"

candidates = [
    p for p in glob.glob("/kaggle/**/Conv_TasNet_Pytorch/train.py", recursive=True)
    if "/working/" in p or "/input/" in p
]
candidates.sort(key=lambda p: (0 if "/working/" in p else 1, p))

if candidates:
    repo_pytorch_src = os.path.dirname(candidates[0])
    repo_root_src    = os.path.dirname(repo_pytorch_src)
    print("Tim thay repo co san tai:", repo_root_src)
    if "/input/" in repo_root_src:
        if not os.path.isdir(REPO_ROOT_WRITABLE):
            print(f"Repo dang o /kaggle/input (read-only) -> copy sang {REPO_ROOT_WRITABLE} ...")
            shutil.copytree(repo_root_src, REPO_ROOT_WRITABLE)
    else:
        REPO_ROOT_WRITABLE = repo_root_src
else:
    if not os.path.isdir(f"{REPO_ROOT_WRITABLE}/Conv_TasNet_Pytorch"):
        print("Khong tim thay repo co san -> git clone tu GitHub ...")
        subprocess.run(["git", "clone", "--depth", "1", GITHUB_REPO_URL, REPO_ROOT_WRITABLE], check=True)
    else:
        print(f"{REPO_ROOT_WRITABLE} da co san, bo qua clone.")

REPO_ROOT = REPO_ROOT_WRITABLE
REPO      = f"{REPO_ROOT}/Conv_TasNet_Pytorch"

ckpt_candidates = glob.glob(f"{REPO_ROOT}/*.pt") + glob.glob(f"{REPO_ROOT}/**/*.pt", recursive=True)
if ckpt_candidates:
    CKPT_SRC = ckpt_candidates[0]
    print("Dung checkpoint co san:", CKPT_SRC)
else:
    CKPT_SRC = f"{REPO_ROOT}/Conv-TasNet_EN.pt"
    print("Khong thay checkpoint co san -> tai ve tu Hugging Face ...")
    subprocess.run(["wget", "-q", "-O", CKPT_SRC, CKPT_URL], check=True)
    print("Da tai checkpoint ve:", CKPT_SRC)

TRAIN_NOISE_RAW = "/kaggle/input/datasets/khabiphmhng/noise-speech/NOISE SPEECH/TRAIN/NOISE"
TRAIN_CLEAN_RAW = "/kaggle/input/datasets/khabiphmhng/noise-speech/NOISE SPEECH/TRAIN/CLEAN"
TEST_RAW        = "/kaggle/input/datasets/khabiphmhng/noise-speech/NOISE SPEECH/TEST "

print("\nREPO       =", REPO, "| ton tai:", os.path.isdir(REPO))
print("CKPT_SRC   =", CKPT_SRC, "| ton tai:", os.path.isfile(CKPT_SRC))
print("DATA8K_DIR =", DATA8K_DIR)


In [ ]:
for p in sorted(glob.glob("/kaggle/input/**/NOISE SPEECH", recursive=True)):
    print(p)
    for root, dirs, files in os.walk(p):
        depth = root[len(p):].count(os.sep)
        if depth <= 2:
            print("  " + root)


In [ ]:
!pip install librosa soundfile pyyaml -q


In [ ]:
import subprocess

os.chdir(REPO)

pairs = [
    (TRAIN_NOISE_RAW, f"{DATA8K_DIR}/TRAIN_NOISE"),
    (TRAIN_CLEAN_RAW, f"{DATA8K_DIR}/TRAIN_CLEAN"),
    (TEST_RAW,        f"{DATA8K_DIR}/TEST "),
] 
for src, dst in pairs:
    print(f"Resample: {src} -> {dst}")
    subprocess.run(["python3", "resample_to_8k.py", src, dst, "--sr", "8000"], check=True)


In [ ]:
import soundfile as sf
import numpy as np

NOISE_DIR = f"{DATA8K_DIR}/TRAIN_NOISE"
CLEAN_DIR = f"{DATA8K_DIR}/TRAIN_CLEAN"
RESID_DIR = f"{DATA8K_DIR}/TRAIN_NOISE_RESIDUAL"
os.makedirs(RESID_DIR, exist_ok=True)

def list_wavs(folder):
    d = {}
    for root, _, names in os.walk(folder):
        for n in names:
            if n.lower().endswith(".wav"):
                d[n] = os.path.join(root, n)   
    return d

noise_files = list_wavs(NOISE_DIR)
clean_files = list_wavs(CLEAN_DIR)
common = sorted(set(noise_files) & set(clean_files))
missing = set(noise_files) - set(clean_files)

print(f"So cap khop nhau: {len(common)}  (NOISE: {len(noise_files)}, CLEAN: {len(clean_files)})")
if missing:
    print(f"CANH BAO: {len(missing)} file NOISE khong co CLEAN tuong ung, se bi bo qua.")

for key in common:
    noisy, sr1 = sf.read(noise_files[key])
    clean, sr2 = sf.read(clean_files[key])
    assert sr1 == 8000 and sr2 == 8000, f"Sample rate khong khop cho {key}: {sr1} vs {sr2}"
    L = min(len(noisy), len(clean))
    residual = noisy[:L] - clean[:L]
    sf.write(os.path.join(RESID_DIR, key), residual, 8000, subtype="PCM_16")

print("Da tao xong residual noise cho", len(common), "file, luu tai:", RESID_DIR)


In [ ]:
import random

random.seed(0)
common_shuffled = common.copy()
random.shuffle(common_shuffled)

val_n = max(1, int(len(common_shuffled) * 0.05))
val_ids = common_shuffled[:val_n]
train_ids = common_shuffled[val_n:]

resid_files = list_wavs(RESID_DIR)

def write_scp(path, ids, mapping):
    with open(path, "w") as f:
        for k in ids:
            f.write(f"{k} {mapping[k]}\n")

write_scp(f"{SCP_DIR}/tr_mix.scp", train_ids, noise_files)
write_scp(f"{SCP_DIR}/tr_s1.scp",  train_ids, clean_files)
write_scp(f"{SCP_DIR}/tr_s2.scp",  train_ids, resid_files)
write_scp(f"{SCP_DIR}/cv_mix.scp", val_ids, noise_files)
write_scp(f"{SCP_DIR}/cv_s1.scp",  val_ids, clean_files)
write_scp(f"{SCP_DIR}/cv_s2.scp",  val_ids, resid_files)

print(f"Train: {len(train_ids)} file, Val: {len(val_ids)} file")
print("Cac file scp:", sorted(os.listdir(SCP_DIR)))


In [ ]:
import shutil

ckpt_dst_dir = f"{RESUME_ROOT}/{CKPT_NAME}"
os.makedirs(ckpt_dst_dir, exist_ok=True)
shutil.copy(CKPT_SRC, f"{ckpt_dst_dir}/best.pt")
print("Da copy checkpoint pretrain vao:", f"{ckpt_dst_dir}/best.pt")


In [ ]:
train_yml_content = f'''#### Conv-TasNet Setting (auto-generated cho finetune)
name: Conv_Tasnet
gpu_ids: [0]
epochs: 20

#### Dataset Configure
datasets:
  num_workers: 2
  batch_size: 8
  fs: 8000
  chunk_len: 4
  chunk_size: 32000
  train:
    mix_scp: {SCP_DIR}/tr_mix.scp
    ref_scp:
      - {SCP_DIR}/tr_s1.scp
      - {SCP_DIR}/tr_s2.scp
    sr: 8000
  val:
    mix_scp: {SCP_DIR}/cv_mix.scp
    ref_scp:
      - {SCP_DIR}/cv_s1.scp
      - {SCP_DIR}/cv_s2.scp
    sr: 8000

#### training settings: learning rate scheme, loss
train:
  optimizer: adam
  min_lr: !!float 1e-8
  patience: 2
  factor: 0.5
  logging_period: 50
  clip_norm: 200
  num_epochs: 20
  checkpoint: {CKPT_NAME}

optimizer_kwargs:
  lr: !!float 1e-4
  weight_decay: !!float 1e-5

#### network configure (KHOP voi checkpoint pretrain, khong sua)
net_conf:
  N: 512
  L: 16
  B: 128
  H: 512
  P: 3
  X: 8
  R: 3
  norm: gln
  num_spks: 2
  activate: relu
  causal: false

#### resume model
resume:
  path: {RESUME_ROOT}
  resume_state: true
'''

yml_path = f"{REPO}/options/train/train.yml"
with open(yml_path, "w") as f:
    f.write(train_yml_content)

print("Da ghi", yml_path)
print(train_yml_content)


In [ ]:
import re

trainer_py = f"{REPO}/trainer.py"
with open(trainer_py) as f:
    content = f.read()

# gỡ bỏ mọi chỗ truyền tham số verbose=... (vd ", verbose=True" hoặc "verbose=True,")
new_content = re.sub(r",?\s*verbose\s*=\s*[A-Za-z0-9_\.]+", "", content)

if new_content != content:
    with open(trainer_py, "w") as f:
        f.write(new_content)
    print("Đã patch trainer.py: bỏ tham số 'verbose' (không còn hỗ trợ trong bản PyTorch mới).")
else:
    print("Không thấy 'verbose=' trong trainer.py, không cần patch.")

In [ ]:
dataloaders_py = f"{REPO}/DataLoaders.py"
with open(dataloaders_py) as f:
    content = f.read()

marker = "    def _collate(self, batch):"
if "def __len__" not in content.split("class DataLoaders")[1]:
    len_method = "    def __len__(self):\n        return len(self.dataset)\n\n"
    content = content.replace(marker, len_method + marker, 1)
    with open(dataloaders_py, "w") as f:
        f.write(content)
    print("Đã thêm __len__ vào class DataLoaders.")
else:
    print("DataLoaders đã có __len__, không cần patch.")

In [ ]:
import re

yml_path = f"{REPO}/options/train/train.yml"
with open(yml_path) as f:
    content = f.read()

content = re.sub(r"epochs:\s*\d+", "epochs: 100", content)
content = re.sub(r"num_epochs:\s*\d+", "num_epochs: 100", content)

with open(yml_path, "w") as f:
    f.write(content)

print("Da sua num_epochs/epochs thanh 90 (checkpoint dang o epoch 70 -> se train them 20 epoch)")

In [ ]:
os.chdir(REPO)
!python train.py --opt options/train/train.yml


## Inference (tach enhanced speech tren tap test)

Cac cell duoi day dung `Separation.py` co san trong repo Conv-TasNet (`Conv_TasNet_Pytorch/Separation.py`) de chay inference tren tap TEST (8kHz), sau do resample nguoc ket qua ve 16kHz de dua vao `volume.py` / `metrics.py` trong pipeline danh gia cua PESEM-VS.

In [ ]:
# 1) Tao file .scp cho tap TEST (8kHz) - dung lai ham list_wavs/write_scp da dinh nghia o tren
TEST_8K_DIR = f"{DATA8K_DIR}/TEST "  # thu muc nay da duoc resample_to_8k.py tao san o cell resample

assert os.path.isdir(TEST_8K_DIR), (
    f"Khong thay {TEST_8K_DIR}. Hay kiem tra lai cell resample_to_8k.py (pairs) da chay chua."
)

test_files = list_wavs(TEST_8K_DIR)
test_ids = sorted(test_files.keys())
print(f"So file test (8kHz): {len(test_ids)}")

write_scp(f"{SCP_DIR}/tt_mix.scp", test_ids, test_files)
print("Da ghi", f"{SCP_DIR}/tt_mix.scp")


In [ ]:
# 2) Chay inference (Separation.py) bang checkpoint vua finetune
SEP_OUT_DIR = f"{WORK_DIR}/separated_8k"
os.makedirs(SEP_OUT_DIR, exist_ok=True)

finetuned_ckpt = f"{REPO}/{CKPT_NAME}/best.pt"
assert os.path.isfile(finetuned_ckpt), (
    f"Khong tim thay checkpoint da finetune tai {finetuned_ckpt}. "
    "Hay chay xong cell train.py o tren truoc."
)

os.chdir(REPO)
subprocess.run([
    "python3", "Separation.py",
    "-mix_scp", f"{SCP_DIR}/tt_mix.scp",
    "-yaml", f"{REPO}/options/train/train.yml",
    "-model", finetuned_ckpt,
    "-gpuid", "0",
    "-save_path", SEP_OUT_DIR,
], check=True)

print("Da tach xong. spk1 = giong noi (enhanced), spk2 = phan con lai (noise). Luu tai:", SEP_OUT_DIR)


In [ ]:
# 3) Resample enhanced (spk1) tu 8kHz ve 16kHz, giu nguyen ten file goc
import librosa
import soundfile as sf

ENHANCED_DIR = f"{WORK_DIR}/enhanced_ConvTasNet"
os.makedirs(ENHANCED_DIR, exist_ok=True)

spk1_dir = f"{SEP_OUT_DIR}/spk1"
spk1_files = sorted(f for f in os.listdir(spk1_dir) if f.lower().endswith(".wav"))

for fname in spk1_files:
    audio_8k, sr = sf.read(os.path.join(spk1_dir, fname), dtype="float32")
    if audio_8k.ndim > 1:
        audio_8k = audio_8k.mean(axis=1)
    audio_16k = librosa.resample(audio_8k, orig_sr=sr, target_sr=16000)
    sf.write(os.path.join(ENHANCED_DIR, fname), audio_16k, 16000)

print(f"Da resample {len(spk1_files)} file ve 16kHz, luu tai: {ENHANCED_DIR}")
print("Buoc tiep theo: chay volume.py de RMS-normalize ve -16 dBFS, roi metrics.py de tinh PESQ/STOI/F0-RMSE/PFR.")
